# Part 2 — Domain-Level SEO Panel (2023–Present)

**Endpoint:** `GET /v1/domain/overview/history`  
**Outputs:** `data/processed/domain_seo_panel_wide.csv` and `domain_seo_panel_long.csv`

In [1]:
import sys, json, time
from pathlib import Path
import requests
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from config import (
    API_BASE, HEADERS, RATE_LIMIT_DELAY,
    ALL_DOMAINS, DOMAIN_META, SOURCE,
    DATA_RAW, DATA_PROCESSED
)

def api_get(path, params=None):
    url = f"{API_BASE}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
    time.sleep(RATE_LIMIT_DELAY)
    return resp

In [2]:
# ── Subscription / credit check ────────────────────────────────────────────────
sub = api_get("/v1/account/subscription").json().get("subscription_info", {})
status   = sub.get("status", "unknown")
expires  = sub.get("expiraton_date", "unknown")
units    = sub.get("units_left", 0)

print(f"Subscription status : {status}")
print(f"Expiration date     : {expires}")
print(f"Units left          : {units}")

if status != "active":
    print(f"\n⚠️  Subscription is '{status}' — data endpoints will return 402.")
    print("   Renew at https://seranking.com/api then re-run this notebook.")

Subscription status : active
Expiration date     : 2026-11-10 19:25:58
Units left          : 99909849.0


## 1. Fetch historical data for all 8 domains

In [3]:
all_records = []
raw_cache   = {}

for domain in ALL_DOMAINS:
    meta = DOMAIN_META[domain]
    print(f"Fetching: {domain}")
    resp = api_get("/v1/domain/overview/history",
                   params={"source": SOURCE, "domain": domain, "type": "organic"})
    records = resp.json() if resp.status_code == 200 else []
    if resp.status_code != 200:
        print(f"  ERROR {resp.status_code}: {resp.text[:150]}")
    raw_cache[domain] = records
    print(f"  → {len(records)} monthly records")
    for r in records:
        r["domain"]    = domain
        r["category"]  = meta["category"]
        r["size_tier"] = meta["size_tier"]
    all_records.extend(records)

raw_path = DATA_RAW / "domain_history_raw.json"
raw_path.write_text(json.dumps(raw_cache, indent=2))
print(f"\nTotal records: {len(all_records)} | Raw saved: {raw_path}")

Fetching: coca-cola.com


  → 72 monthly records
Fetching: pepsi.com


  → 72 monthly records
Fetching: drpepper.com


  → 72 monthly records
Fetching: drinkolipop.com


  → 72 monthly records
Fetching: mcdonalds.com


  → 72 monthly records
Fetching: wendys.com


  → 72 monthly records
Fetching: shakeshack.com


  → 72 monthly records
Fetching: fiveguys.com


  → 72 monthly records

Total records: 576 | Raw saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/raw/domain_history_raw.json


## 2. Build panel dataframe

In [4]:
if not all_records:
    print("⚠️  No records — renew subscription and re-run.")
    df = pd.DataFrame()
else:
    df = pd.DataFrame(all_records).copy()
    df["date"] = pd.to_datetime(
        df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2) + "-01"
    )
    df = df[df["date"] >= "2023-01-01"].drop(columns=["year","month"])
    df = df.sort_values(["category","domain","date"]).reset_index(drop=True)
    print(f"Panel (wide): {df.shape}")
    print(f"Date range  : {df['date'].min().date()} → {df['date'].max().date()}")
    print(f"Domains     : {sorted(df['domain'].unique())}")
    display(df.head(10))

Panel (wide): (304, 12)
Date range  : 2023-01-01 → 2026-02-01
Domains     : ['coca-cola.com', 'drinkolipop.com', 'drpepper.com', 'fiveguys.com', 'mcdonalds.com', 'pepsi.com', 'shakeshack.com', 'wendys.com']


,keywords_count,traffic_sum,top1_5,top6_10,top11_20,top21_50,top51_100,price_sum,domain,category,size_tier,date
0,107446,306206,15377,7928,13994,31867,48094,284218.0,coca-cola.com,beverages,large,2023-01-01
1,99264,324547,15316,7482,13045,29339,43751,407931.0,coca-cola.com,beverages,large,2023-02-01
2,95925,337790,15570,7564,12617,27894,42118,385610.0,coca-cola.com,beverages,large,2023-03-01
3,97628,301804,15548,7616,12729,28468,42931,323535.0,coca-cola.com,beverages,large,2023-04-01
4,98477,283304,15549,7844,13013,29162,42535,308172.0,coca-cola.com,beverages,large,2023-05-01
5,99798,268216,15518,7767,12971,29483,43507,309778.0,coca-cola.com,beverages,large,2023-06-01
6,101172,253171,15329,7476,12750,29156,45782,1412596.0,coca-cola.com,beverages,large,2023-07-01
7,104500,220854,15328,7555,12569,30078,48381,1856145.0,coca-cola.com,beverages,large,2023-08-01
8,130327,236028,18631,8797,13748,36272,64073,2414936.0,coca-cola.com,beverages,large,2023-09-01
9,140739,283623,19797,9282,14025,38801,71146,2606728.0,coca-cola.com,beverages,large,2023-10-01


In [5]:
if not df.empty:
    cov = df.groupby(["category","size_tier","domain"])["date"].agg(
        months="count", earliest="min", latest="max"
    ).reset_index()
    cov["earliest"] = cov["earliest"].dt.strftime("%Y-%m")
    cov["latest"]   = cov["latest"].dt.strftime("%Y-%m")
    print("Coverage per domain:")
    display(cov)

Coverage per domain:


,category,size_tier,domain,months,earliest,latest
0,beverages,large,coca-cola.com,38,2023-01,2026-02
1,beverages,large,drpepper.com,38,2023-01,2026-02
2,beverages,large,pepsi.com,38,2023-01,2026-02
3,beverages,small,drinkolipop.com,38,2023-01,2026-02
4,fast_food,large,mcdonalds.com,38,2023-01,2026-02
5,fast_food,large,wendys.com,38,2023-01,2026-02
6,fast_food,small,fiveguys.com,38,2023-01,2026-02
7,fast_food,small,shakeshack.com,38,2023-01,2026-02


## 3. Save datasets

In [6]:
if not df.empty:
    # Wide format
    wide_path = DATA_PROCESSED / "domain_seo_panel_wide.csv"
    df.to_csv(wide_path, index=False)
    print(f"Saved (wide): {wide_path}")

    # Long format
    metric_cols = [c for c in
        ["keywords_count","traffic_sum","price_sum",
         "top1_5","top6_10","top11_20","top21_50","top51_100"]
        if c in df.columns]
    id_cols = ["domain","category","size_tier","date"]
    df_long = df[id_cols + metric_cols].melt(
        id_vars=id_cols, value_vars=metric_cols,
        var_name="metric", value_name="value"
    )
    long_path = DATA_PROCESSED / "domain_seo_panel_long.csv"
    df_long.to_csv(long_path, index=False)
    print(f"Saved (long): {long_path} — {len(df_long)} rows")
    display(df_long.head(10))
else:
    print("⚠️  Nothing to save.")

Saved (wide): /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/domain_seo_panel_wide.csv
Saved (long): /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/domain_seo_panel_long.csv — 2432 rows


,domain,category,size_tier,date,metric,value
0,coca-cola.com,beverages,large,2023-01-01,keywords_count,107446.0
1,coca-cola.com,beverages,large,2023-02-01,keywords_count,99264.0
2,coca-cola.com,beverages,large,2023-03-01,keywords_count,95925.0
3,coca-cola.com,beverages,large,2023-04-01,keywords_count,97628.0
4,coca-cola.com,beverages,large,2023-05-01,keywords_count,98477.0
5,coca-cola.com,beverages,large,2023-06-01,keywords_count,99798.0
6,coca-cola.com,beverages,large,2023-07-01,keywords_count,101172.0
7,coca-cola.com,beverages,large,2023-08-01,keywords_count,104500.0
8,coca-cola.com,beverages,large,2023-09-01,keywords_count,130327.0
9,coca-cola.com,beverages,large,2023-10-01,keywords_count,140739.0


## 4. Quick summary statistics

In [7]:
if not df.empty:
    # Latest month snapshot
    latest = df[df["date"] == df["date"].max()].sort_values("traffic_sum", ascending=False)
    show = [c for c in ["domain","category","size_tier","keywords_count","traffic_sum","price_sum"] if c in latest.columns]
    print(f"Latest snapshot ({df['date'].max().strftime('%Y-%m')}):")
    display(latest[show])

    # Growth from earliest to latest
    t0 = df[df["date"] == df["date"].min()]
    t1 = df[df["date"] == df["date"].max()]
    growth = t0[["domain","traffic_sum","keywords_count"]].merge(
        t1[["domain","traffic_sum","keywords_count"]], on="domain", suffixes=("_start","_end")
    )
    for col in ["traffic_sum","keywords_count"]:
        growth[f"{col}_pct"] = (
            (growth[f"{col}_end"] - growth[f"{col}_start"]) /
            growth[f"{col}_start"].replace(0, float("nan")) * 100
        ).round(1)
    print(f"\nGrowth {df['date'].min().strftime('%Y-%m')} → {df['date'].max().strftime('%Y-%m')}:")
    display(growth)
else:
    print("⚠️  No data — renew SE Ranking Data API subscription and re-run.")

Latest snapshot (2026-02):


,domain,category,size_tier,keywords_count,traffic_sum,price_sum
303,wendys.com,fast_food,large,298105,8751735,2171569.32
227,mcdonalds.com,fast_food,large,914858,5144264,1392712.75
189,fiveguys.com,fast_food,small,170029,1608632,107540.30
265,shakeshack.com,fast_food,small,922854,968296,1322635.06
37,coca-cola.com,beverages,large,206053,371020,91663.77
113,drpepper.com,beverages,large,13746,89325,29393.96
75,drinkolipop.com,beverages,small,15690,25382,2391.60
151,pepsi.com,beverages,large,21885,18255,3545.13



Growth 2023-01 → 2026-02:


,domain,traffic_sum_start,keywords_count_start,traffic_sum_end,keywords_count_end,traffic_sum_pct,keywords_count_pct
0,coca-cola.com,306206,107446,371020,206053,21.2,91.8
1,drinkolipop.com,42820,12408,25382,15690,-40.7,26.5
2,drpepper.com,86909,5346,89325,13746,2.8,157.1
3,pepsi.com,42032,45701,18255,21885,-56.6,-52.1
4,fiveguys.com,384434,131531,1608632,170029,318.4,29.3
5,mcdonalds.com,6475286,961963,5144264,914858,-20.6,-4.9
6,shakeshack.com,236514,322886,968296,922854,309.4,185.8
7,wendys.com,2828846,285278,8751735,298105,209.4,4.5
